In [1]:
# ==========================================
# CELDA 1: Setup
# ==========================================
import sys, os

!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git

PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

!pip install -q transformers torch scikit-learn polars joblib optuna

print("✓ Entorno configurado")


Cloning into 'proyecto-transformacion-texto-imagen'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 312 (delta 7), reused 0 (delta 0), pack-reused 296 (from 1)
Receiving objects: 100% (312/312), 6.13 MiB | 10.67 MiB/s, done.
Resolving deltas: 100% (157/157), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 16.0 MB/s eta 0:00:00
✓ Entorno configurado


In [4]:
# ==========================================
# CELDA 2: Carga y generación de corpus filtrado
# ==========================================
import polars as pl
import numpy as np
from features.pandemic_stopwords import apply_pandemic_filter, count_pandemic_terms

# Cargar datos originales
train = pl.read_csv(f'{PROJECT_ROOT}/data/processed/train.csv')
test  = pl.read_csv(f'{PROJECT_ROOT}/data/processed/test.csv')
val = pl.read_csv(f'{PROJECT_ROOT}/data/processed/validation.csv')

X_train_orig = train.get_column('text').to_list()
y_train      = train.get_column('manual_classification').to_numpy()
X_test_orig  = test.get_column('text').to_list()
y_test       = test.get_column('manual_classification').to_numpy()
X_val_orig  = val.get_column('text').to_list()
y_val       = val.get_column('manual_classification').to_numpy()

# Aplicar filtro de pandemia
X_train_nopand = apply_pandemic_filter(X_train_orig)
X_test_nopand  = apply_pandemic_filter(X_test_orig)
X_val_nopand = apply_pandemic_filter(X_val_orig)

# ── Estadística del filtro ──────────────────────────────────
n_afectados_train = sum(1 for t in X_train_orig if count_pandemic_terms(t) > 0)
n_afectados_test  = sum(1 for t in X_test_orig  if count_pandemic_terms(t) > 0)

print(f"Textos con términos de pandemia:")
print(f"  Train: {n_afectados_train}/{len(X_train_orig)} ({n_afectados_train/len(X_train_orig)*100:.1f}%)")
print(f"  Test:  {n_afectados_test}/{len(X_test_orig)}  ({n_afectados_test/len(X_test_orig)*100:.1f}%)")
print(f"  Val: {n_afectados_test}/{len(X_val_orig)}  ({n_afectados_test/len(X_val_orig)*100:.1f}%)")

# Guardar corpus filtrado para reutilización futura
train_nopand = train.with_columns(pl.Series('text', X_train_nopand))
test_nopand  = test.with_columns(pl.Series('text', X_test_nopand))
val_nopand = val.with_columns(pl.Series('text', X_val_nopand))

train_nopand.write_csv(f'{PROJECT_ROOT}/data/processed/train_nopandemic.csv')
test_nopand.write_csv(f'{PROJECT_ROOT}/data/processed/test_nopandemic.csv')
val_nopand.write_csv(f'{PROJECT_ROOT}/data/processed/validation_nopandemic.csv')

print("\n✓ Corpus filtrado guardado en data/processed/")


Textos con términos de pandemia:
  Train: 223/908 (24.6%)
  Test:  23/114  (20.2%)
  Val: 23/114  (20.2%)

✓ Corpus filtrado guardado en data/processed/


In [5]:
import os
import json
import torch
import numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

# ==============================
# Reproducibilidad
# ==============================
set_seed(42)


# ==============================
# Dataset
# ==============================
class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }


# ==============================
# Métricas
# ==============================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


# ==============================
# Fine-Tuning Principal
# ==============================
def run_finetuning(train_texts, train_labels, val_texts, val_labels, model_name):

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    train_ds = DepressionDataset(train_texts, train_labels, tokenizer)
    val_ds = DepressionDataset(val_texts, val_labels, tokenizer)

    safe_model_name = model_name.replace("/", "_")
    output_dir = f"models/checkpoints/{safe_model_name}"

    training_args = TrainingArguments(
        output_dir=output_dir,

        # Estrategia de evaluación
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        # Optimización
        learning_rate=2.3048016342698774e-05,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.06379717552110609,
        warmup_ratio=0.0013564417879888579,

        # Selección del mejor modelo
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,

        # Control de checkpoints
        save_total_limit=1,
        fp16=True,

        # Reproducibilidad
        seed=42,
        report_to='none',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    # ==========================
    # Evaluación Final
    # ==========================
    final_metrics = trainer.evaluate()
    print("\nFinal Evaluation Metrics:")
    print(final_metrics)

    # Guardar métricas en JSON
    os.makedirs("results", exist_ok=True)
    with open(f"results/{safe_model_name}_metrics.json", "w") as f:
        json.dump(final_metrics, f, indent=4)

    # ==========================
    # Matriz de Confusión
    # ==========================
    predictions = trainer.predict(val_ds)
    preds = np.argmax(predictions.predictions, axis=1)
    cm = confusion_matrix(val_labels, preds)

    print("\nConfusion Matrix:")
    print(cm)

    return trainer

In [6]:
# ==========================================
# CELDA 3: Fine-tuning BETO sin términos de pandemia
# (Mismos hiperparámetros óptimos del Exp 4.1)
# ==========================================

print("="*55)
print("EXP 4.2: BETO Fine-Tuning SIN términos de pandemia")
print("="*55)

trainer_nopand = run_finetuning(
    X_train_nopand, y_train,
    X_val_nopand,  y_val,
    "dccuchile/bert-base-spanish-wwm-cased"
)

print("\n✅ Exp 4.2 completado")


EXP 4.2: BETO Fine-Tuning SIN términos de pandemia


config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Final Evaluation Metrics:
{'eval_loss': 0.5443912744522095, 'eval_accuracy': 0.6842105263157895, 'eval_precision_macro': 0.6668831168831169, 'eval_recall_macro': 0.6668831168831169, 'eval_f1_macro': 0.6668831168831169, 'eval_runtime': 0.3945, 'eval_samples_per_second': 288.993, 'eval_steps_per_second': 20.28, 'epoch': 3.0}

Confusion Matrix:
[[52 18]
 [18 26]]

✅ Exp 4.2 completado


In [9]:
# ==========================================
# Guardar tokenizer en el checkpoint
# ==========================================
from transformers import AutoTokenizer

CKPT_PATH = f'{PROJECT_ROOT}/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-57'
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"

# Cargar tokenizer desde HuggingFace y guardarlo junto al checkpoint
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(CKPT_PATH)

# Verificar
import os
archivos = os.listdir(CKPT_PATH)
print("📁 Archivos en checkpoint:")
for a in sorted(archivos):
    tam = os.path.getsize(os.path.join(CKPT_PATH, a)) / 1e6
    print(f"  ✅ {a:<40} {tam:.2f} MB")


📁 Archivos en checkpoint:
  ✅ config.json                              0.00 MB
  ✅ model.safetensors                        439.43 MB
  ✅ optimizer.pt                             878.99 MB
  ✅ rng_state.pth                            0.01 MB
  ✅ scaler.pt                                0.00 MB
  ✅ scheduler.pt                             0.00 MB
  ✅ tokenizer.json                           0.73 MB
  ✅ tokenizer_config.json                    0.00 MB
  ✅ trainer_state.json                       0.00 MB
  ✅ training_args.bin                        0.01 MB


In [10]:
from transformers import AutoModelForSequenceClassification

# Definir el mapeo correcto según los datos
id2label = {0: "no_depresivo", 1: "depresivo"}
label2id = {"no_depresivo": 0, "depresivo": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    CKPT_PATH,
    id2label=id2label,
    label2id=label2id
)

# Guardar el modelo con el mapeo correcto
model.save_pretrained(CKPT_PATH)
print("✓ Modelo guardado con mapeo de etiquetas correcto")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Modelo guardado con mapeo de etiquetas correcto


In [11]:
# ==========================================
# Extraer embeddings del modelo fine-tuneado
# y guardar PKL compatible con tu pipeline
# ==========================================
import torch
import numpy as np
import joblib
import os
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

CKPT_PATH = f'{PROJECT_ROOT}/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-57'
MODELS_DIR  = f'{PROJECT_ROOT}/models/best_model'
os.makedirs(MODELS_DIR, exist_ok=True)

def extraer_embeddings_finetuned(texts, ckpt_path, batch_size=16):
    """
    Extrae embeddings [CLS] del BETO fine-tuneado.
    Usa AutoModel (sin cabeza de clasificación) para obtener representaciones puras.
    """
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(ckpt_path)

    # AutoModel carga SOLO el encoder, sin la capa de clasificación
    model     = AutoModel.from_pretrained(ckpt_path).to(device)
    model.eval()

    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Extrayendo embeddings"):
        batch  = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs    = model(**inputs)
            cls_emb    = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_emb)

    return np.vstack(all_embeddings)

# Extraer embeddings
print("Generando embeddings del modelo fine-tuneado...")
X_train_emb = extraer_embeddings_finetuned(X_train_nopand, CKPT_PATH)
X_test_emb  = extraer_embeddings_finetuned(X_test_nopand,  CKPT_PATH)

print(f"\n✓ Shape train: {X_train_emb.shape}")  # Esperado: (908, 768)
print(f"✓ Shape test:  {X_test_emb.shape}")     # Esperado: (114, 768)

# Guardar PKL con estructura compatible con tu pipeline
embeddings_pkg = {
    'X_train':      X_train_emb,
    'y_train':      y_train,
    'X_test':       X_test_emb,
    'y_test':       y_test,
    'model_name':   'BETO_HPO_finetuned',
    'checkpoint':   CKPT_PATH,
    'shape':        X_train_emb.shape[1],  # 768 (dimensión del embedding)
}

pkl_path = os.path.join(MODELS_DIR, 'Exp_4.2_BETO_NoPandemic_embeddings.pkl')
joblib.dump(embeddings_pkg, pkl_path)
print(f"\n✅ PKL guardado en: {pkl_path}")

# Verificar carga
loaded = joblib.load(pkl_path)
print(f"✓ Verificación PKL: X_train={loaded['X_train'].shape}, X_test={loaded['X_test'].shape}")


Generando embeddings del modelo fine-tuneado...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/proyecto-transformacion-texto-imagen/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-57
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extrayendo embeddings: 100%|██████████| 57/57 [00:07<00:00,  8.12it/s]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/proyecto-transformacion-texto-imagen/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-57
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Extrayendo embeddings: 100%|██████████| 8/8 [00:00<00:00,  8.79it/s]


✓ Shape train: (908, 768)
✓ Shape test:  (114, 768)

✅ PKL guardado en: /content/proyecto-transformacion-texto-imagen/models/best_model/Exp_4.2_BETO_NoPandemic_embeddings.pkl
✓ Verificación PKL: X_train=(908, 768), X_test=(114, 768)


In [12]:
# ==========================================
# Verificación del checkpoint guardado
# ==========================================
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

CKPT_PATH = f'{PROJECT_ROOT}/models/checkpoints/dccuchile_bert-base-spanish-wwm-cased/checkpoint-57'

# 2. Cargar y hacer una predicción de prueba
print("\n🔍 Prueba de carga del modelo:")
try:
    tokenizer = AutoTokenizer.from_pretrained(CKPT_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(CKPT_PATH)
    model.eval()

    # Prueba rápida
    textos_prueba = [
        "Me siento sin esperanza, todo está oscuro",  # Esperado: depresivo (1)
        "Hoy fue un gran día, estoy muy feliz"         # Esperado: no depresivo (0)
    ]
    inputs = tokenizer(textos_prueba, padding=True, truncation=True,
                       max_length=128, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=1).tolist()

    etiquetas = {0: 'No depresivo', 1: 'Depresivo'}
    print("  Predicciones de prueba:")
    for texto, pred in zip(textos_prueba, preds):
        print(f"    [{etiquetas[pred]}] → '{texto[:45]}...'")
    print("\n✅ Checkpoint cargado y funcional")

except Exception as e:
    print(f"❌ Error: {e}")



🔍 Prueba de carga del modelo:


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Predicciones de prueba:
    [Depresivo] → 'Me siento sin esperanza, todo está oscuro...'
    [No depresivo] → 'Hoy fue un gran día, estoy muy feliz...'

✅ Checkpoint cargado y funcional


In [13]:
# ==========================================
# CELDA : Comparación de resultados
# ==========================================
import joblib, json, os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from tqdm import tqdm

# ── Exp 4.1: Cargar desde PKL existente (sin checkpoint) ───
print("Cargando embeddings Exp 4.1 (desde PKL)...")
pkg_41         = joblib.load(f'{PROJECT_ROOT}/models/best_model/Exp_4.1_BETO_HPO_embeddings.pkl')
X_train_emb_41 = pkg_41['X_train']
X_test_emb_41  = pkg_41['X_test']
print(f"✓ Cargado | Shape train: {X_train_emb_41.shape} | test: {X_test_emb_41.shape}")

# ── Exp 4.2: Extraer desde el checkpoint recién guardado ───
print("Cargando embeddings Exp 4.2 (desde PKL)...")
pkg_42         = joblib.load(f'{PROJECT_ROOT}/models/best_model/Exp_4.2_BETO_NoPandemic_embeddings.pkl')
X_train_emb_42 = pkg_42['X_train']
X_test_emb_42  = pkg_42['X_test']
print(f"✓ Cargado | Shape train: {X_train_emb_42.shape} | test: {X_test_emb_42.shape}")

# ── Comparación con LogisticRegression ─────────────────────
resultados = {}
for nombre, X_tr, X_te in [
    ('4.1 CON pandemia', X_train_emb_41, X_test_emb_41),
    ('4.2 SIN pandemia', X_train_emb_42, X_test_emb_42),
]:
    clf = LogisticRegression(class_weight='balanced',
                             solver='liblinear', random_state=42)
    clf.fit(X_tr, y_train)
    preds = clf.predict(X_te)
    f1    = f1_score(y_test, preds, average='macro')
    resultados[nombre] = {'f1_macro': round(f1, 4)}

print("\n" + "="*50)
print("📊 COMPARACIÓN FINAL - Sesgo de Pandemia")
print("="*50)
print(f"{'Experimento':<30} {'F1-Macro':>10} {'vs Baseline':>12}")
print("-"*55)
print(f"{'Baseline (sin HPO)':<30} {'0.7411':>10}")
for nombre, res in resultados.items():
    delta = res['f1_macro'] - 0.7411
    print(f"{'Exp ' + nombre:<30} {res['f1_macro']:>10.4f}  ({delta:+.4f})")

# Guardar JSON en results/
comp = {
    'baseline':              0.7411,
    'exp_4.1_con_pandemia':  resultados['4.1 CON pandemia']['f1_macro'],
    'exp_4.2_sin_pandemia':  resultados['4.2 SIN pandemia']['f1_macro'],
    'delta_4.1_vs_baseline': round(resultados['4.1 CON pandemia']['f1_macro'] - 0.7411, 4),
    'delta_4.2_vs_baseline': round(resultados['4.2 SIN pandemia']['f1_macro'] - 0.7411, 4),
    'delta_4.1_vs_4.2':      round(resultados['4.1 CON pandemia']['f1_macro'] -
                                   resultados['4.2 SIN pandemia']['f1_macro'], 4),
}
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)
with open(f'{PROJECT_ROOT}/results/comparacion_sesgo_pandemia.json', 'w') as f:
    json.dump(comp, f, indent=4)

print(f"\n✓ Resultados guardados en results/comparacion_sesgo_pandemia.json")


Cargando embeddings Exp 4.1 (desde PKL)...
✓ Cargado | Shape train: (908, 768) | test: (114, 768)
Cargando embeddings Exp 4.2 (desde PKL)...
✓ Cargado | Shape train: (908, 768) | test: (114, 768)

📊 COMPARACIÓN FINAL - Sesgo de Pandemia
Experimento                      F1-Macro  vs Baseline
-------------------------------------------------------
Baseline (sin HPO)                 0.7411
Exp 4.1 CON pandemia               0.7881  (+0.0470)
Exp 4.2 SIN pandemia               0.7348  (-0.0063)

✓ Resultados guardados en results/comparacion_sesgo_pandemia.json


In [ ]:

# ==========================================
# CELDA 7: Backup en Google Drive
# ==========================================
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/TT_Resultados/HPO'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copiar resultados
shutil.copytree(RESULTS_DIR, f'{DRIVE_DIR}/optuna', dirs_exist_ok=True)
shutil.copytree(FINAL_OUTPUT_DIR, f'{DRIVE_DIR}/modelo_final', dirs_exist_ok=True)

print(f"✓ Backup completo en Google Drive: {DRIVE_DIR}")
